In [0]:
from pyspark.sql import functions as F
 
BRONZE_LAYER_PATH = "/Volumes/workspace/default/ecommerce_medallion_pipeline/bronze/"
SILVER_LAYER_PATH = "/Volumes/workspace/default/ecommerce_medallion_pipeline/silver/"
CHECKPOINT_PATH = "/Volumes/workspace/default/ecommerce_medallion_pipeline/checkpoint/"

# Exploratory Data Analysis

## Customers

In [0]:
customers_bronze = spark.readStream.format("delta").load(f"{BRONZE_LAYER_PATH}customers")

# in EDA readStream() will not work with (printing/show the data bec, show() works with static data only)

# customers_bronze = spark.read.format("delta").load(f"{BRONZE_LAYER_PATH}customers")
# this will be commente in production

# customers_bronze.printSchema()

# what is the diffrence between show() and display()
# display() shows the table as HTML format, better UI, you can scroll V and H and filter and sort
# display() is the best for EDA and it is a databrics function not for all python libraries
# display(customers_bronze)
# customers_bronze.show()

In [0]:
# display(customers_bronze.select("Email").distinct().orderBy("Email"))
# display(customers_bronze.select("Gender").distinct().distinct())

# Group by every column in the dataframe
# duplicate_rows = customers_bronze.groupBy(customers_bronze.columns) \
#     .count() \
#     .filter(F.col("count") > 1)

# Display the duplicate rows and their repetition count
# duplicate_rows.show(truncate=False)

In [0]:
# from the schema all are strin 
# and we want to change these data types
# root
#  |-- CustomerID: string to int
#  |-- RegistrationDate: string to date
#  |-- _rescued_data: string (nullable = true)


# also gender has many value we want to cintralize them
# Male --> M 1 MALE male
# Feale --> F 0 FEMALE female
# all values in Gender
# MALE
# Female
# male
# 1
#  Male 
#  Female 
# Male
# m
# F
# 0
# f
# M
# FEMALE
# female

# also RegistrationDate have many data format we want to cintralize to dd-MM-yyyy

# also email have many nulls and missing vlues and we want to make it first_name.last_name@gamil.com
# the missing values in email
# |                   -|
# |                 N/A|
# |                NULL|
# |                 NaN|
# |                None|
 
 
gender_normalized = F.trim(F.lower(F.col("Gender"))) 

email_wrong_values = ["-" ,"N/A" ,"NULL" ,"NaN" ,"None"]

parsed_date = F.coalesce(
    F.try_to_date(F.col("RegistrationDate"), "yyyy-MM-dd"),
    F.try_to_date(F.col("RegistrationDate"), "dd/MM/yyyy"),
    F.try_to_date(F.col("RegistrationDate"), "MM-dd-yyyy"),
    F.try_to_date(F.col("RegistrationDate"), "dd MMM yyyy"),
    F.try_to_date(F.col("RegistrationDate"), "yyyy/MM/dd"),
    F.try_to_date(F.col("RegistrationDate"), "MMMM dd, yyyy")
)

email_missing = F.col("Email").isNull() | F.lower(F.trim(F.col("Email"))).isin("", "-", "n/a", "null", "nan", "none")

customers_cleaned = (
    customers_bronze
    .dropDuplicates(["CustomerID"])
    .withColumn("CustomerID", F.col("CustomerID").cast("bigint"))
    # .withColumn("RegistrationDate", F.to_date(F.col("RegistrationDate"), "dd-MM-yyyy")) 
    # if we have many data format will convert then to null only this format "dd-MM-yyyy" will convert to date
    .withColumn("RegistrationDate", parsed_date) 
    .withColumn("Gender", 
                F.when(gender_normalized.isin("male","m","1"), "Male")
                .when(gender_normalized.isin("female","f","0"), "Female")
                .otherwise("unknown"))

    .withColumn("is_email_generated", F.when(email_missing, "Yes").otherwise("No"))
    .withColumn("Email", F.when(email_missing,
            F.lower(F.concat(F.col("FirstName"), F.lit("."), F.col("LastName"), F.lit("@example.com"))))
            .otherwise(F.col("Email")))
    )


# display(customers_cleaned)
# customers_cleaned.printSchema()

## Categories



In [0]:
categories_bronze = spark.readStream.format("delta").load(f"{BRONZE_LAYER_PATH}categories")
# in EDA readStream() will not work with (printing/show the data bec, show() works with static data only)

# categories_bronze = spark.read.format("delta").load(f"{BRONZE_LAYER_PATH}categories")

# categories_bronze.printSchema()

# what is the diffrence between show() and display()
# display() shows the table as HTML format, better UI, you can scroll V and H and filter and sort
# display() is the best for EDA and it is a databrics function not for all python libraries
# display(categories_bronze)
# categories_bronze.show()

In [0]:
# justo we need to change the category id to int 
categories_cleaned = (
    categories_bronze
    .withColumn("CategoryID", F.col("CategoryID").cast("int"))
    .withColumn("MainCategory",
        F.when(F.col("CategoryName").isin("Clothing", "Shoes", "Jewelry"), "Fashion")
        .when(F.col("CategoryName").isin("Home & Kitchen", "Furniture", "Garden", "Tools"), "Home & Improvement")
        .when(F.col("CategoryName").isin("Electronics", "Office Supplies"), "Tech & Office")
        .when(F.col("CategoryName").isin("Groceries", "Health", "Beauty"), "Essentials & Wellness")
        .when(F.col("CategoryName").isin("Books", "Movies", "Music"), "Media")
        .when(F.col("CategoryName").isin("Baby Products", "Toys", "Pet Supplies", "Sports"), "Lifestyle & Family")
        .when(F.col("CategoryName").isin("Automotive"), "Automotive")
        .otherwise("Other"))
)
# categories_cleaned.printSchema()
# display(categories_cleaned)

## Departments

In [0]:
departments_bronze = spark.readStream.format("delta").load(f"{BRONZE_LAYER_PATH}departments")
# in EDA readStream() will not work with (printing/show the data bec, show() works with static data only)

# departments_bronze = spark.read.format("delta").load(f"{BRONZE_LAYER_PATH}departments")

# departments_bronze.printSchema()

# what is the diffrence between show() and display()
# display() shows the table as HTML format, better UI, you can scroll V and H and filter and sort
# display() is the best for EDA and it is a databrics function not for all python libraries
# display(departments_bronze)
# departments_bronze.show()

In [0]:
# justo we need to change the category id to int 
departments_cleaned = departments_bronze.withColumn("DepartmentID", F.col("DepartmentID").cast("int"))

# departments_cleaned.printSchema()

## Employees

In [0]:
employees_bronze = spark.readStream.format("delta").load(f"{BRONZE_LAYER_PATH}employees")
# in EDA readStream() will not work with (printing/show the data bec, show() works with static data only)

# employees_bronze = spark.read.format("delta").load(f"{BRONZE_LAYER_PATH}employees")

# employees_bronze.printSchema()

# what is the diffrence between show() and display()
# display() shows the table as HTML format, better UI, you can scroll V and H and filter and sort
# display() is the best for EDA and it is a databrics function not for all python libraries
# display(employees_bronze)
# employees_bronze.show()

In [0]:
 # all ids to int
 # salary to float
 # HireDate to data
 # also make the gender like the customers


# like we did in customer so just we need to change "RegistrationDate" to "HireDate"
# why not to make a function that takes the name of the column and make it parsed date with this all formate

from pyspark.sql import functions as F

def parse_my_date(date_col_name):
    return F.coalesce(
        F.try_to_date(F.col(date_col_name), "yyyy-MM-dd"),
        F.try_to_date(F.col(date_col_name), "dd/MM/yyyy"),
        F.try_to_date(F.col(date_col_name), "MM-dd-yyyy"),
        F.try_to_date(F.col(date_col_name), "dd MMM yyyy"),
        F.try_to_date(F.col(date_col_name), "yyyy/MM/dd"),
        F.try_to_date(F.col(date_col_name), "MMMM dd, yyyy")
    )


# like we did in customer we do not have to write it again
# gender_normalized = F.trim(F.lower(F.col("Gender")))



employees_cleaned = (
    employees_bronze
    .dropDuplicates()
    .withColumn("EmployeeID", F.col("EmployeeID").cast("int"))
    # try_cast() bec, cast() can not cast NULL as string to interger so, try cast if is can not cast will return NULL
    .withColumn("ManagerID", F.col("ManagerID").try_cast("int")) 
    .withColumn("DepartmentID", F.col("DepartmentID").cast("int"))
    .withColumn("Gender", F.when(gender_normalized.isin("male","m","1"), "Male")
                            .when(gender_normalized.isin("female","f","0"), "Female"))
    # try_cast() bec, cast() can not cast NULL as string to interger so, try cast if is can not cast will return NULL
    # but the problem that out salary contains "," eg. 12,000
    # so, try_cast() will convert it to NULL and this is not what we want
    # so, we need to remove the "," first
    # translate(srcCol, matching, replace)
    .withColumn("Salary", F.translate(F.col("Salary"), "$, ","").try_cast("decimal(18,2)"))
    .withColumn("HireDate", parse_my_date("HireDate"))
)

# now we have some nulls in salary

# employees_cleaned.printSchema()
# display(employees_cleaned)

## Order Details

In [0]:
order_details_bronze = spark.readStream.format("delta").load(f"{BRONZE_LAYER_PATH}order_details")
# in EDA readStream() will not work with (printing/show the data bec, show() works with static data only)

# order_details_bronze = spark.read.format("delta").load(f"{BRONZE_LAYER_PATH}order_details")

# order_details_bronze.printSchema()

# what is the diffrence between show() and display()
# display() shows the table as HTML format, better UI, you can scroll V and H and filter and sort
# display() is the best for EDA and it is a databrics function not for all python libraries
# display(order_details_bronze)
# order_details_bronze.show()

In [0]:
# orderis and details id to bigint
# product id and quantity to int
# unitprice and discount to float

order_details_cleaned = (
    order_details_bronze
    .dropDuplicates()
    .withColumn("OrderDetailID", F.col("OrderDetailID").cast("bigint"))
    .withColumn("OrderID", F.col("OrderID").cast("bigint"))
    .withColumn("ProductID", F.col("ProductID").cast("int"))
    .withColumn("Quantity", F.col("Quantity").cast("int"))
    .withColumn("UnitPrice", F.col("UnitPrice").cast("decimal(18,2)"))
    # we have 5 and 0.05 in the discount so we cintralize them to be fraction
    # .withColumn("Discount",
    #     F.when(F.col("Discount") > 1, F.col("Discount") / 100)
    #     .otherwise(F.col("Discount"))
    #     .cast("decimal(8,2)"))
    
    # this cast so we can check if the value is grater then 1
    .withColumn("Discount", F.col("Discount").cast("float"))
    .withColumn("Discount",
        F.when(F.col("Discount") > 1, F.col("Discount") / 100)
        .otherwise(F.col("Discount")))
    # this cast to make the discont 3 nums only eg. 0.15 - 0.00 - 0.10 - 0.05
    .withColumn("Discount", F.col("Discount").cast("decimal(3,2)"))
    .withColumn("TotalSales", (F.col("UnitPrice") * F.col("Quantity") * (1 - F.col("Discount"))).cast("decimal(18,2)"))

)


# order_details_cleaned.printSchema()
# display(order_details_cleaned)

## Orders

In [0]:
orders_bronze = spark.readStream.format("delta").load(f"{BRONZE_LAYER_PATH}orders")
# in EDA readStream() will not work with (printing/show the data bec, show() works with static data only)

# orders_bronze = spark.read.format("delta").load(f"{BRONZE_LAYER_PATH}orders")

# orders_bronze.printSchema()

# what is the diffrence between show() and display()
# display() shows the table as HTML format, better UI, you can scroll V and H and filter and sort
# display() is the best for EDA and it is a databrics function not for all python libraries
# display(orders_bronze)
# orders_bronze.show()

In [0]:
 # CustomerID and SalesEmployeeID to int 
 # OrderID to bigint
 # Status clean it and cintralize it
 # OrderDate to data

# +---------+
# |   Status|
# +---------+
# |Delivered|
# |  Shipped|
# |  Pending|
# |Cancelled|
# +---------+

orders_cleaned = (
    orders_bronze
    .dropDuplicates()
    .withColumn("OrderID", F.col("OrderID").cast("bigint"))
    # try_cast() bec, cast() can not cast NULL as string to interger so, try cast if is can not cast will return NULL
    .withColumn("CustomerID", F.col("CustomerID").try_cast("int")) 
    .withColumn("SalesEmployeeID", F.col("SalesEmployeeID").cast("int"))
    # initcap() = title() Mo Bedier --> Mo Bedier
    .withColumn("Status",F.trim(F.initcap(F.col("Status"))))
    .withColumn("Status",
                F.when(F.col("Status").isin("Cancelled", "Canceled"), "Cancelled")
                .otherwise(F.col("Status")))
    .withColumn("OrderDate", parse_my_date("OrderDate"))
)

# orders_cleaned.printSchema()
# display(orders_cleaned)

In [0]:
# orders_cleaned.select("Status").distinct().show()

## Payments

In [0]:
payments_bronze = spark.readStream.format("delta").load(f"{BRONZE_LAYER_PATH}payments")
# in EDA readStream() will not work with (printing/show the data bec, show() works with static data only)

# payments_bronze = spark.read.format("delta").load(f"{BRONZE_LAYER_PATH}payments")

# payments_bronze.printSchema()

# what is the diffrence between show() and display()
# display() shows the table as HTML format, better UI, you can scroll V and H and filter and sort
# display() is the best for EDA and it is a databrics function not for all python libraries
# display(payments_bronze)
# payments_bronze.show()

In [0]:
# PaymentID and OrderID to bigint
# PaymentMethod and PaymentStatus cintralization
# PaymentDate to data
# Amount to float


# +-------------+
# |PaymentStatus|
# +-------------+
# |    Completed|
# |     Declined|
# |         Done|
# |        Error|
# |       Failed|
# |      Success|
# |    completed|
# |       failed|
# +-------------+

# +----------------+
# |   PaymentMethod|
# +----------------+
# |   Bank Transfer|
# |           C.O.D|
# |              CC|
# |             COD|
# |            Cash|
# |Cash on Delivery|
# |     Credit Card|
# |      MasterCard|
# |          PayPal|
# |            Visa|
# |            cash|
# | cash on delevry|
# |             cod|
# |     credit card|
# |          paypal|
# +----------------+


payments_cleaned = (
    payments_bronze
    .dropDuplicates()
    .withColumn("PaymentID", F.col("PaymentID").cast("bigint"))
    .withColumn("OrderID", F.col("OrderID").cast("bigint"))
    .withColumn("Amount", F.translate(F.col("Amount"),"$, " ,"").cast("decimal(10,2)")) 
    .withColumn("PaymentStatus", F.trim(F.lower(F.col("PaymentStatus"))))
    .withColumn("PaymentStatus", 
            F.when(F.col("PaymentStatus").isin("done", "completed", "success"), "Completed")
            .when(F.col("PaymentStatus").isin("error", "declined", "failed"), "Failed")
            .otherwise(F.lit(None)))
    .withColumn("PaymentMethod", F.trim(F.lower(F.col("PaymentMethod"))))
    .withColumn("PaymentMethod", 
            F.when(F.col("PaymentMethod").isin("cod", "c.o.d", "cash" , "cash on delevry" , "cash on delivery"), "Cash on Delivery")
            .when(F.col("PaymentMethod").isin("mastercard" , "visa" , "credit card", "creditcard", "cc"), "Payment Card")
            .when(F.col("PaymentMethod").isNotNull(), F.initcap(F.col("PaymentMethod")))
            .otherwise(F.lit(None)))
    .withColumn("PaymentDate", parse_my_date("PaymentDate"))
)



# payments_cleaned.printSchema()
# display(payments_cleaned)

## Product Suppliers

In [0]:
product_suppliers_bronze = spark.readStream.format("delta").load(f"{BRONZE_LAYER_PATH}product_suppliers")
# in EDA readStream() will not work with (printing/show the data bec, show() works with static data only)

# product_suppliers_bronze = spark.read.format("delta").load(f"{BRONZE_LAYER_PATH}product_suppliers")

# product_suppliers_bronze.printSchema()

# what is the diffrence between show() and display()
# display() shows the table as HTML format, better UI, you can scroll V and H and filter and sort
# display() is the best for EDA and it is a databrics function not for all python libraries
# display(product_suppliers_bronze)
# product_suppliers_bronze.show()

In [0]:
product_suppliers_cleaned = (
    product_suppliers_bronze
    .dropDuplicates()
    .withColumn("ProductID", F.col("ProductID").cast("int"))
    .withColumn("SupplierID", F.col("SupplierID").cast("int")) 
)

# product_suppliers_cleaned.printSchema()
# display(product_suppliers_cleaned)

## Products

In [0]:
products_bronze = spark.readStream.format("delta").load(f"{BRONZE_LAYER_PATH}products")

# products_bronze = spark.read.format("delta").load(f"{BRONZE_LAYER_PATH}products")

# products_bronze.printSchema()

# what is the diffrence between show() and display()
# display() shows the table as HTML format, better UI, you can scroll V and H and filter and sort
# display() is the best for EDA and it is a databrics function not for all python libraries
# display(products_bronze)
# products_bronze.show()

In [0]:
attributes_schema = "warranty_months INT, color STRING"

products_cleaned = (
    products_bronze
    .dropDuplicates(["ProductID"])
    .withColumn("ProductID", F.col("ProductID").cast("bigint"))
    .withColumn("CategoryID", F.col("CategoryID").cast("int"))
    .withColumn("Price", F.translate(F.col("Price"), "$, ", "").try_cast("decimal(10,2)"))
    .withColumn("Cost", F.translate(F.col("Cost"), "$, ", "").try_cast("decimal(10,2)"))
    .withColumn("Stock", F.col("Stock").try_cast("int"))
    .withColumn("StockType", 
                    F.when(F.col("Stock").isNull(), "Unknown") 
                    .when(F.col("Stock") < 0, "Invalid") 
                    .when(F.col("Stock") == 0, "No Stock") 
                    .when(F.col("Stock") <= 20, "Low Stock")                             
                    .when(F.col("Stock") <= 120, "normal Stock")                           
                    .otherwise("High Stock"))
    .withColumn("attr", F.from_json(F.regexp_replace(F.col("Attributes"), "'", '"'),  attributes_schema))
    .withColumn("warranty_months", F.col("attr.warranty_months"))
    .withColumn("color", F.col("attr.color"))
    .drop("Attributes", "attr")
)



# products_cleaned.printSchema()
# display(products_cleaned)

## Shipments

In [0]:
shipments_bronze = spark.readStream.format("delta").load(f"{BRONZE_LAYER_PATH}shipments")

# shipments_bronze = spark.read.format("delta").load(f"{BRONZE_LAYER_PATH}shipments")

# shipments_bronze.printSchema()

# what is the diffrence between show() and display()
# display() shows the table as HTML format, better UI, you can scroll V and H and filter and sort
# display() is the best for EDA and it is a databrics function not for all python libraries
# display(shipments_bronze)
# shipments_bronze.show()

In [0]:
# ShipmentID and OrderID to bigint
# ShipperID to int
# ShipDate and DeliveryDate to data


shipments_cleaned = (
    shipments_bronze
    .dropDuplicates()
    .withColumn("ShipmentID", F.col("ShipmentID").cast("bigint"))
    .withColumn("OrderID", F.col("OrderID").cast("bigint"))
    .withColumn("ShipperID", F.col("ShipperID").cast("int"))
    .withColumn("ShipDate", parse_my_date("ShipDate"))
    .withColumn("DeliveryDate", parse_my_date("DeliveryDate"))
    .withColumn("DaysToArrive",F.datediff(F.col("DeliveryDate") , F.col("ShipDate")))
    .withColumn("DeliveryStatus", 
                F.when(F.col("DaysToArrive").isNull(), "Not Delivered")
                .when(F.col("DaysToArrive") < 0, "Invalid Dates")
                .otherwise("Delivered")
                )
)



# shipments_cleaned.printSchema()
# display(shipments_cleaned)

## Shippers

In [0]:
shippers_bronze = spark.readStream.format("delta").load(f"{BRONZE_LAYER_PATH}shippers")

# shippers_bronze = spark.read.format("delta").load(f"{BRONZE_LAYER_PATH}shippers")

# shippers_bronze.printSchema()

# what is the diffrence between show() and display()
# display() shows the table as HTML format, better UI, you can scroll V and H and filter and sort
# display() is the best for EDA and it is a databrics function not for all python libraries
# display(shippers_bronze)
# shippers_bronze.show()

In [0]:
shippers_cleaned = (
    shippers_bronze
    .dropDuplicates()
    .withColumn("ShipperID", F.col("ShipperID").cast("int"))
    .withColumn("ShipperType", 
            F.when(F.col("CompanyName").isin("USPS" , "Royal Mail" , "China Post"), "National Postal")
            .when(F.col("CompanyName").isin("DHL" , "FedEx" , "UPS" , "TNT"), "Global Express")
            .when(F.col("CompanyName").isin("Aramex" , "SF Express"), "Regional Specialists")
            .when(F.col("CompanyName").isin("Amazon Logistics"), "Captive E-commerce")
            .otherwise("Unknown")
            )
)



# shippers_cleaned.printSchema()
# display(shippers_cleaned)

## Suppliers

In [0]:
suppliers_bronze = spark.readStream.format("delta").load(f"{BRONZE_LAYER_PATH}suppliers")
# suppliers_bronze = spark.read.format("delta").load(f"{BRONZE_LAYER_PATH}suppliers")

# suppliers_bronze.printSchema()

# what is the diffrence between show() and display()
# display() shows the table as HTML format, better UI, you can scroll V and H and filter and sort
# display() is the best for EDA and it is a databrics function not for all python libraries
# display(suppliers_bronze)
# suppliers_bronze.show()

In [0]:
suppliers_cleaned = (
    suppliers_bronze
    .dropDuplicates()
    .withColumn("SupplierID", F.col("SupplierID").cast("int"))
)

# suppliers_cleaned.printSchema()
# display(suppliers_cleaned)

## Orders CDC Incremental

In [0]:
orders_cdc_incremental_bronze = spark.readStream.format("delta").load(f"{BRONZE_LAYER_PATH}orders_cdc_incremental")
# orders_cdc_incremental_bronze = spark.read.format("delta").load(f"{BRONZE_LAYER_PATH}orders_cdc_incremental")

# orders_cdc_incremental_bronze.printSchema()

# what is the diffrence between show() and display()
# display() shows the table as HTML format, better UI, you can scroll V and H and filter and sort
# display() is the best for EDA and it is a databrics function not for all python libraries
# display(orders_cdc_incremental_bronze)
# orders_cdc_incremental_bronze.show()

In [0]:
from pyspark.sql.window import Window


cdc_window = Window.partitionBy("OrderID").orderBy(F.col("EventTimestamp").desc())

orders_cdc_incremental_cleaned = (
    orders_cdc_incremental_bronze
    .dropDuplicates()
    .withColumn("OrderID", F.col("OrderID").try_cast("bigint"))
    .withColumn("Status",F.trim(F.initcap(F.col("Status"))))
    .withColumn("Status",
                F.when(F.col("Status").isin("Cancelled", "Canceled"), "Cancelled")
                .otherwise(F.col("Status")))
    .withColumn("EventTimestamp", F.col("EventTimestamp").try_cast("timestamp"))
    .filter((F.col("OrderID").isNotNull()) & (F.col("Status").isNotNull()) 
            & (F.col("EventTimestamp").isNotNull()))
    # to get the last change
    .withColumn("rn", F.row_number().over(cdc_window))
    .filter(F.col("rn") == 1)
)

# orders_cdc_incremental_cleaned.printSchema()
# display(orders_cdc_incremental_cleaned)

load to silver (upsert or insert)

In [ ]:
from delta.tables import DeltaTable

def upsert_to_silver(df, table_name, key_cols, insert_only=False):
    table_path = f"{SILVER_LAYER_PATH}{table_name}"
    # the join bec, in product suppliers we have many key so we could not made like this
    # .merge(df.alias("source"), f"target.{key_col} = source.{key_col}")
    merge_condition = " AND ".join(f"target.{k} = source.{k}" for k in key_cols)

    # if the table is not stored yet -> just load it
    if not DeltaTable.isDeltaTable(spark, table_path):
        df.write.format("delta").save(table_path)

    # if the table already existed (upsert or insert)
    else:
        delta_table = DeltaTable.forPath(spark, table_path)

        merge = delta_table.alias("target").merge(df.alias("source"), merge_condition)

        if not insert_only:
            # if existed update it
            merge = merge.whenMatchedUpdateAll()

        # if not just insert it
        merge.whenNotMatchedInsertAll().execute()


upsert_to_silver(customers_cleaned, "customers", ["CustomerID"])
upsert_to_silver(categories_cleaned, "categories", ["CategoryID"])
upsert_to_silver(departments_cleaned, "departments", ["DepartmentID"])
upsert_to_silver(employees_cleaned, "employees", ["EmployeeID"])
upsert_to_silver(order_details_cleaned, "order_details", ["OrderDetailID"])
# insert only for orders bec, if the shipments changed we will do it from the cdc file
upsert_to_silver(orders_cleaned, "orders", ["OrderID"], insert_only=True)  
upsert_to_silver(payments_cleaned, "payments", ["PaymentID"])
# here is the many keys
upsert_to_silver(product_suppliers_cleaned, "product_suppliers", ["ProductID", "SupplierID"]) 
upsert_to_silver(products_cleaned, "products", ["ProductID"])
upsert_to_silver(shipments_cleaned, "shipments", ["ShipmentID"])
upsert_to_silver(shippers_cleaned, "shippers", ["ShipperID"])
upsert_to_silver(suppliers_cleaned, "suppliers", ["SupplierID"])

In [ ]:
CDC

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.window import Window

def process_cdc_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        return
    
    cdc_window = Window.partitionBy("OrderID").orderBy(F.col("EventTimestamp").desc())
    
    cleaned = (batch_df
            .dropDuplicates()
            .withColumn("OrderID", F.col("OrderID").try_cast("bigint"))
            .withColumn("Status",F.trim(F.initcap(F.col("Status"))))
            .withColumn("Status",
                        F.when(F.col("Status").isin("Cancelled", "Canceled"), "Cancelled")
                        .otherwise(F.col("Status")))
            .withColumn("EventTimestamp", F.col("EventTimestamp").try_cast("timestamp"))
            .filter((F.col("OrderID").isNotNull()) & (F.col("Status").isNotNull()) 
                    & (F.col("EventTimestamp").isNotNull()))
            # to get the last change
            .withColumn("rn", F.row_number().over(cdc_window))
            .filter(F.col("rn") == 1)
    )
    
    target_table = DeltaTable.forPath(spark, f"{SILVER_LAYER_PATH}orders")
    (target_table.alias("target")
     .merge(cleaned.alias("source"), "target.OrderID = source.OrderID")
     .whenMatchedUpdate(set = {"Status": "source.Status"})
     .execute())

(
    spark.readStream.format("delta").load(f"{BRONZE_LAYER_PATH}orders_cdc_incremental")
    .writeStream
    .foreachBatch(process_cdc_batch)
    .option("checkpointLocation", f"{CHECKPOINT_PATH}silver_orders_cdc")
    .trigger(availableNow=True)
    .start()
).awaitTermination()